> **中文阅读版**：本文件是 `03_ib_paper_trading_demo.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# Interactive Brokers 模拟交易演示

**章节**：25 - 实盘交易系统
**章节参考**：25.2（与 Interactive Brokers 集成）
**学习成果**：LO2 - 将 ml4t-backtest 策略连接到实盘券商

**Docker 镜像**：`ml4t`（要求 IB TWS/Gateway 在主机端口 7497 上运行）

本 notebook 演示：
1. 连接到 IB TWS 模拟交易账户
2. 查询账户信息和持仓
3. 请求历史数据用于策略预热
4. 带 tick 聚合的实时数据馈送
5. 在影子模式（shadow mode）下安全提交订单

**学习目标**：
- 在实盘引擎启动前验证 IB 连接性、账户状态和预热数据。
- 在影子模式（shadow mode）的实盘工作流中复用同一回测策略。
- 演练显式失败（loud-fail）的设置检查清单：当 TWS/Gateway 不可达或市场休市时，notebook 不会根据历史数据合成实盘行为。

**前提条件**：
- IB TWS 或 Gateway 正在运行并已启用 API
- 模拟交易端口：7497（TWS）或 4002（Gateway）
- 将 `IB_ACCOUNT` 环境变量设置为你的 IB 模拟账户 ID（例如 `DU1234567`）；如果未设置，notebook 会向 `IBBroker` 传入 `None`，这会允许 broker 在会话中选取默认账户。
- 熟悉实盘交易章节中贯穿使用的 ETF 动量示例

In [1]:
"""Connect ml4t strategies to IB with shadow-mode risk controls."""

import os

`MARKET_DATA_TYPE` 决定 TWS 返回哪种报价。将其保留为 `None` 会沿用会话所配置的类型，当模拟账户带有实时 Level 1 订阅时，这是正确选择。没有这些订阅的模拟账户会以 “No market data available...” 拒绝 MARKET 订单，因此本 notebook 改为请求延迟报价。TWS 接受的四个值为：1 表示实时，2 表示冻结，3 表示延迟，4 表示延迟冻结。

In [2]:
# Production defaults. Papermill may inject overrides for CI.
IB_HOST = "127.0.0.1"
IB_PORT = 7497  # Paper trading port
CLIENT_ID = 10  # Use unique ID per notebook
ACCOUNT = os.environ.get(
    "IB_ACCOUNT"
)  # set IB_ACCOUNT=DU... before running; None picks the session default
MARKET_DATA_TYPE: int | None = 3  # delayed quotes; see above
SYMBOLS = ["SPY", "QQQ", "IWM"]  # ETFs to monitor
WARMUP_DAYS = 10  # Must provide at least lookback + 1 daily closes
LIVE_DURATION_SECONDS = 75

In [3]:
import asyncio
import logging
import warnings
from datetime import datetime

os.environ.setdefault("NUMEXPR_MAX_THREADS", "16")

from async_utils import run_async

from utils.paths import display_path, get_output_dir

In [4]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

# Reduce noise from ib_async
logging.getLogger("ib_async").setLevel(logging.WARNING)
logging.getLogger("ml4t.live.brokers.ib").setLevel(logging.WARNING)

In [5]:
# Import ml4t.live components
from ml4t.backtest import OrderSide, Strategy
from ml4t.live import LiveEngine, LiveRiskConfig
from ml4t.live.brokers.ib import IBBroker
from ml4t.live.feeds import BarAggregator, IBDataFeed
from ml4t.live.safety import SafeBroker

print("[OK] ml4t.live components imported successfully")

[OK] ml4t.live components imported successfully


导入检查会在尝试任何 Interactive Brokers 连接之前运行，因此缺少包的环境与无法连接 TWS 的会话会以不同方式失败。将两者合并为一个不透明错误，正是导致实盘部署调试缓慢的原因。

## 1. 连接到 Interactive Brokers

TWS/Gateway 必须正在运行，并且：
- 已启用 API 访问（Edit > Global Configuration > API > Settings）
- 已设置套接字端口（模拟交易默认：7497）
- 已勾选 “Enable ActiveX and Socket Clients”

In [6]:
# Create IBBroker instance
broker = IBBroker(
    host=IB_HOST,
    port=IB_PORT,
    client_id=CLIENT_ID,
    account=ACCOUNT,
    market_data_type=MARKET_DATA_TYPE,
)

print(f"IBBroker configured for {IB_HOST}:{IB_PORT}")
print(f"Account selection: {'configured paper account' if ACCOUNT else 'session default'}")
print(f"Client ID: {CLIENT_ID}")

IBBroker configured for 127.0.0.1:7497
Account selection: session default
Client ID: 10


主机、端口、账户和客户端 ID 会被打印出来，而不是作为常量留在上方单元格中，因为账户或会话不匹配是最常见的实盘失败之一，并且在某些内容已经被提交到错误位置之前往往不可见。


In [7]:
async def connect_to_ib():
    """Connect to IB and show account summary."""
    print("\n" + "=" * 60)
    print("CONNECTING TO INTERACTIVE BROKERS")
    print("=" * 60)

    await broker.connect()

    account = str(broker._account or "")
    if not account.startswith("DU"):
        await broker.disconnect()
        raise RuntimeError("Refusing to continue: connected IB account is not a paper account")

    print("\n[OK] Connected to IB")
    print("   Paper account identity: [OK]")
    # Get account values
    nlv = await broker.get_account_value_async()
    cash = await broker.get_cash_async()

    print("\nACCOUNT READINESS")
    print("   Account values received: [OK]")

    # Get positions
    positions = await broker.get_positions_async()
    print(f"   Open positions received: {len(positions)}")

    return nlv, cash, positions

In [8]:
# Run connection
try:
    warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
    nlv, cash, positions = run_async(connect_to_ib())
except Exception as exc:
    print()
    print("=" * 60)
    print("ERROR: IB paper session unreachable")
    print("=" * 60)
    print(f"Could not connect to {IB_HOST}:{IB_PORT}")
    print(f"Underlying error: {type(exc).__name__}: {exc}")
    print()
    print("Setup checklist:")
    print("  1. Start TWS or IB Gateway and log into a paper account.")
    print("  2. Edit -> Global Configuration -> API -> Settings:")
    print("     - 'Enable ActiveX and Socket Clients' must be checked.")
    print(f"     - Socket port must be {IB_PORT} (TWS paper=7497, Gateway paper=4002).")
    print("     - 127.0.0.1 must be in 'Trusted IPs', or 'Read-Only API' unchecked.")
    print("  3. Confirm no other client is using this client_id.")
    print()
    print("Re-run this notebook once TWS is reachable.")
    raise RuntimeError("IB paper session unreachable") from exc


CONNECTING TO INTERACTIVE BROKERS



[OK] Connected to IB
   Paper account identity: [OK]

ACCOUNT READINESS
   Account values received: [OK]
   Open positions received: 0


当 TWS 不可达时，连接代码块会打印检查清单并停止，而不是用占位数据替代并让 notebook 的其余部分看起来像是已经交易过。broker 断开连接是一种运行状态，因此该检查位于信号循环之前，而不是循环内部。

## 2. 请求历史数据

在实盘运行策略之前，我们需要历史数据来：
- 计算初始指标值（移动平均等）
- 为仓位规模（position sizing）建立基线
- 验证数据质量

IB 通过 `reqHistoricalData` 提供历史 bar。

In [9]:
from ib_async import Stock


async def get_historical_data(symbol: str, days: int = 5) -> list[dict]:
    """Request historical daily bars from IB and return OHLCV records."""
    contract = Stock(symbol, "SMART", "USD")
    qualified = await broker.ib.qualifyContractsAsync(contract)
    if not qualified:
        raise RuntimeError(f"IB could not qualify the {symbol} contract")

    # `timeout=0` disables ib_async's internal `asyncio.wait_for` wrapper,
    # which fails under nest_asyncio on Python 3.14.
    bars = await broker.ib.reqHistoricalDataAsync(
        contract,
        endDateTime="",
        durationStr=f"{days} D",
        barSizeSetting="1 day",
        whatToShow="TRADES",
        useRTH=True,
        timeout=0,
    )

    result = []
    for bar in bars:
        result.append(
            {
                "timestamp": bar.date,
                "open": bar.open,
                "high": bar.high,
                "low": bar.low,
                "close": bar.close,
                "volume": bar.volume,
            }
        )

    if not result:
        raise RuntimeError(f"IB returned no warm-up bars for {symbol}")
    return result

In [10]:
# Request historical data for our symbols
print("\n" + "=" * 60)
print("HISTORICAL DATA (Warm-up)")
print("=" * 60)

historical_data = {}
try:
    for symbol in SYMBOLS:
        warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
        bars = run_async(get_historical_data(symbol, WARMUP_DAYS))
        if len(bars) < 6:
            raise RuntimeError(f"{symbol} returned {len(bars)} bars; at least 6 are required")
        historical_data[symbol] = bars
        latest = bars[-1]
        print(f"\n{symbol}: {len(bars)} days of data")
        print(
            f"   Latest: {latest['timestamp']} - Close: ${latest['close']:.2f}, Volume: {latest['volume']:,}"
        )
except Exception:
    warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
    run_async(broker.disconnect())
    raise

assert set(historical_data) == set(SYMBOLS)


HISTORICAL DATA (Warm-up)



SPY: 10 days of data
   Latest: 2026-07-23 - Close: $739.91, Volume: 6,032,786.0



QQQ: 10 days of data
   Latest: 2026-07-23 - Close: $694.94, Volume: 5,368,493.0



IWM: 10 days of data
   Latest: 2026-07-23 - Close: $291.95, Volume: 4,450,643.0


预热会打印指标初始化所用的确切 bar。没有确定性的预热路径，重连或开盘后的最初几分钟就会根据仍在填充的指标来确定仓位规模。

## 3. 策略定义

该策略与我们在回测中使用的策略**完全相同**。
`on_data` 签名同时适用于 `ml4t.backtest.Engine` 和 `ml4t.live.LiveEngine`。

In [11]:
class MomentumStrategy(Strategy):
    """Five-day ETF momentum strategy shared by backtest and live engines."""

    def __init__(self, lookback: int = 5, threshold: float = 0.02):
        self.lookback = lookback
        self.threshold = threshold
        self.prices: dict[str, list[float]] = {}
        self.signals: list[dict] = []

    def on_start(self, broker):
        """Called when engine starts."""
        logger.info(f"Strategy started: Momentum({self.lookback}, {self.threshold})")
        for symbol, bars in historical_data.items():
            self.prices[symbol] = [bar["close"] for bar in bars]
            logger.info(f"  {symbol}: Loaded {len(self.prices[symbol])} historical prices")

    def on_data(self, timestamp: datetime, data: dict, context: dict, broker):
        """Update trailing momentum and route threshold crossings."""
        for symbol, bar in data.items():
            prices = self.prices.setdefault(symbol, [])
            close = bar["close"]
            prices.append(close)
            if len(prices) <= self.lookback:
                continue
            momentum = (close - prices[-self.lookback - 1]) / prices[-self.lookback - 1]
            position = broker.get_position(symbol)
            has_position = position is not None and position.quantity > 0
            side = None
            if momentum > self.threshold and not has_position:
                side = OrderSide.BUY
            elif momentum < -self.threshold and has_position:
                side = OrderSide.SELL
            if side is None:
                continue
            action = side.value.upper()
            self.signals.append(
                {"timestamp": timestamp, "symbol": symbol, "action": action, "momentum": momentum}
            )
            logger.info(f"{action} {symbol}: momentum {momentum:.2%}")
            broker.submit_order(symbol, 100, side=side)

    def on_end(self, broker):
        """Called when engine stops."""
        logger.info(f"Strategy ended. Signals generated: {len(self.signals)}")

## 4. 安全 Broker 配置

在实盘之前，我们用 `SafeBroker` 包装 broker，它提供：
- 影子模式（shadow mode）（虚拟订单经由 `VirtualPortfolio` 路由，绝不会发送到 IB）
- 持仓和订单价值上限、限速、终止开关（kill switch）
- 持久化的 `RiskState`，因此重启后可以读回每日亏损计数器
- 启动对账：`safe_broker.connect()` 会将上一次运行持久化的快照与 broker 当前持仓和挂单进行差异比较

In [12]:
RISK_STATE_PATH = get_output_dir(25, "ib_paper_demo") / "risk_state.json"
risk_config = LiveRiskConfig(
    shadow_mode=True,  # CRITICAL: Virtual orders only!
    max_position_value=50_000.0,
    max_order_value=10_000.0,
    max_orders_per_minute=5,
    max_daily_loss=2_500.0,
    max_data_staleness_seconds=60,
    dedup_window_seconds=0.0,  # Disable for demo
    state_file=str(RISK_STATE_PATH),
)

safe_broker = SafeBroker(broker, risk_config)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
run_async(safe_broker.connect())

print("\n" + "=" * 60)
print("RISK CONFIGURATION (SHADOW MODE)")
print("=" * 60)
print("   Shadow Mode: [OK] ENABLED (no real orders)")
print(f"   Max Position Value: ${risk_config.max_position_value:,.0f}")
print(f"   Max Order Value: ${risk_config.max_order_value:,.0f}")
print(f"   Max Daily Loss: ${risk_config.max_daily_loss:,.0f}")
print(f"   Max Data Staleness: {risk_config.max_data_staleness_seconds}s")
print(f"   Rate Limit: {risk_config.max_orders_per_minute}/minute")
print(f"   Risk State: {display_path(RISK_STATE_PATH)}")
report = safe_broker.reconciliation_report
print(f"   Startup Reconciliation: {'clean' if report and report['clean'] else 'review report'}")

2026-07-23 10:18:23,273 - ml4t.live.safety - INFO - SafeBroker initialized. Shadow mode: True


2026-07-23 10:18:23,276 - ml4t.live.safety - INFO - SafeBroker reconciliation clean



RISK CONFIGURATION (SHADOW MODE)
   Shadow Mode: [OK] ENABLED (no real orders)
   Max Position Value: $50,000
   Max Order Value: $10,000
   Max Daily Loss: $2,500
   Max Data Staleness: 60s
   Rate Limit: 5/minute
   Risk State: <ML4T_OUTPUT_DIR>/ch25_ib_paper_demo/risk_state.json
   Startup Reconciliation: clean


## 5. 实时数据馈送

`IBDataFeed` 从 IB 订阅实时 tick 数据；`BarAggregator` 将这些 tick 滚动聚合为分钟 bar 供策略使用。实盘 notebook 只有在市场开市时才有意义。如果运行在收盘后开始，notebook 会显式失败，而不是用历史 bar 替代（那是披着实盘外衣的回测）。

In [13]:
def _is_rth_now(zone: str = "America/New_York") -> bool:
    """Return True iff wall-clock now is inside US-equity regular trading hours."""
    from zoneinfo import ZoneInfo

    now = datetime.now(ZoneInfo(zone))
    if now.weekday() >= 5:
        return False
    open_t = now.replace(hour=9, minute=30, second=0, microsecond=0)
    close_t = now.replace(hour=16, minute=0, second=0, microsecond=0)
    return open_t <= now <= close_t

数据馈送、聚合器和引擎在一个辅助函数中组装起来，因此下面的有界运行只关注生命周期和清理。

In [14]:
def build_live_stack() -> tuple[BarAggregator, LiveEngine]:
    """Build the IB feed, minute aggregator, and shadow execution engine."""
    ib_feed = IBDataFeed(ib=broker.ib, symbols=SYMBOLS, tick_throttle_ms=1000)
    feed = BarAggregator(source_feed=ib_feed, bar_size_minutes=1, assets=SYMBOLS)
    strategy = MomentumStrategy(lookback=5, threshold=0.02)
    engine = LiveEngine(strategy=strategy, broker=safe_broker, feed=feed)
    return feed, engine

固定时长的 RTH 运行可验证连接性，而不会留下无人值守的策略循环。

In [15]:
async def run_live_demo(duration_seconds: int = 30):
    """Run strategy with live IB data for specified duration."""
    print("\n" + "=" * 60)
    print("LIVE TRADING DEMO (Shadow Mode)")
    print("=" * 60)

    if not _is_rth_now():
        print()
        print("ERROR: Market is closed. This notebook requires an open RTH session.")
        print("US equity RTH: 09:30-16:00 America/New_York, Mon-Fri.")
        print("Re-run during RTH; the notebook will not synthesize live behaviour from history.")
        raise RuntimeError("IB live-feed gate requires an open US-equity RTH session")

    feed, engine = build_live_stack()
    print(f"\nStarting live engine for {duration_seconds} seconds...")
    print("   Watching: " + ", ".join(SYMBOLS))

    engine_task: asyncio.Task | None = None
    try:
        await engine.connect()

        # Run for a fixed duration. `asyncio.wait_for` is unreliable under
        # nest_asyncio in notebook kernels, so we drive the timeout manually.
        engine_task = asyncio.create_task(engine.run())
        await asyncio.sleep(duration_seconds)
        print(f"\nDemo duration ({duration_seconds}s) reached")
    finally:
        if engine_task is not None and not engine_task.done():
            engine_task.cancel()
            try:
                await engine_task
            except asyncio.CancelledError:
                pass
        feed.stop()
        print("\nEngine Statistics:")
        for key, value in engine.stats.items():
            print(f"   {key}: {value}")


# The live workflow runs after the shadow-order routine is defined so one
# outer `finally` block can always release the IB session.

无论实时数据馈送是否启用，策略和风险配置都相同；改变的只是传输层。保持这一边界正是让整个技术栈可测试的原因，也意味着更换数据馈送绝不等于更换交易规则。

## 6. 订单提交演示

我们来演示影子模式（shadow mode）下的订单提交。
订单会被虚拟跟踪，但绝不会发送到 IB。

In [16]:
async def fetch_ib_snapshot(symbol: str) -> float | None:
    """Fetch one delayed top-of-book snapshot from IB for `symbol` and return mid price."""
    contract = Stock(symbol, "SMART", "USD")
    qualified = await broker.ib.qualifyContractsAsync(contract)
    if not qualified:
        return None
    ticker = broker.ib.reqMktData(qualified[0], "", snapshot=True, regulatorySnapshot=False)
    deadline = datetime.now().timestamp() + 5.0
    while datetime.now().timestamp() < deadline:
        bid = float(ticker.bid) if ticker.bid and ticker.bid > 0 else None
        ask = float(ticker.ask) if ticker.ask and ticker.ask > 0 else None
        last = float(ticker.last) if ticker.last and ticker.last > 0 else None
        close = float(ticker.close) if ticker.close and ticker.close > 0 else None
        if bid and ask:
            return (bid + ask) / 2
        if last:
            return last
        if close:
            return close
        await asyncio.sleep(0.2)
    return None

下面每个订单之前，都会先由从 IB 拉取并记录在 `SafeBroker` 上的真实快照报价作为前置，绝不是研究时价格，也绝不是内存中的 mock。这个先后顺序正是本章所教授的操作纪律。延迟数据足以演示这一点；账户是否持有实时订阅是账户层面的问题。

两个数量都按当前 SPY 和 QQQ 的水平调整，以适配 `max_order_value`，因此演示展示的是虚拟成交，而不是触发上限拒绝。其他所有控制仍适用于每一腿。

In [17]:
async def demonstrate_order_submission():
    """Show order submission in shadow mode using live IB snapshot quotes."""
    print("\n" + "=" * 60)
    print("ORDER SUBMISSION DEMO (Shadow Mode)")
    print("=" * 60)

    print("\nFetching delayed snapshot quotes from IB...")
    snapshot_prices = {}
    for symbol in SYMBOLS:
        price = await fetch_ib_snapshot(symbol)
        if price is None:
            raise RuntimeError(f"No IB snapshot quote received for {symbol} within 5 seconds")
        safe_broker.record_market_snapshot(symbol, price)
        snapshot_prices[symbol] = price
        print(f"   {symbol}: ${price:,.2f}")
    assert set(snapshot_prices) == set(SYMBOLS)

    orders = [("SPY", 10), ("QQQ", 12)]

    for symbol, qty in orders:
        print(f"\nSubmitting virtual BUY order: {qty} shares {symbol}")
        order = await safe_broker.submit_order_async(symbol, qty, side=OrderSide.BUY)
        print(f"   Order ID: {order.order_id}")
        print(f"   Status: {order.status.value}")
        print(f"   Side: {order.side.value}")
        print(f"   Quantity: {order.quantity}")

    vp = safe_broker._virtual_portfolio
    print("\nFinal Virtual Portfolio:")
    print(f"   Cash: ${vp.cash:,.2f}")
    for symbol, pos in vp.positions.items():
        value = pos.quantity * (pos.current_price or pos.entry_price)
        print(f"   {symbol}: {pos.quantity} shares @ ${pos.entry_price:.2f} = ${value:,.2f}")

一个外层清理边界同时覆盖实时行情演示和虚拟订单演示。

In [18]:
async def run_shadow_workflow() -> None:
    """Run the live-feed and shadow-order gates with guaranteed cleanup."""
    try:
        await run_live_demo(duration_seconds=LIVE_DURATION_SECONDS)
        await demonstrate_order_submission()
    finally:
        await safe_broker.disconnect()


warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"nest_asyncio")
run_async(run_shadow_workflow())

2026-07-23 10:18:23,371 - ml4t.live.engine - INFO - LiveEngine: Connecting...


2026-07-23 10:18:23,373 - ml4t.live.safety - INFO - SafeBroker reconciliation clean


2026-07-23 10:18:23,374 - ml4t.live.feeds.ib_feed - INFO - IBDataFeed: Starting feed for 3 symbols



LIVE TRADING DEMO (Shadow Mode)

Starting live engine for 75 seconds...
   Watching: SPY, QQQ, IWM


2026-07-23 10:18:23,753 - ml4t.live.feeds.ib_feed - INFO - IBDataFeed: Subscribed to 3 symbols


...

2026-07-23 10:19:39,394 - ml4t.live.safety - INFO - Shadow: Opened SPY LONG 10


2026-07-23 10:19:39,395 - ml4t.live.safety - INFO - SHADOW: buy 10 SPY @ $740.44 (value: $7,404)


2026-07-23 10:19:39,396 - ml4t.live.safety - INFO - Shadow: Opened QQQ LONG 12


2026-07-23 10:19:39,396 - ml4t.live.safety - INFO - SHADOW: buy 12 QQQ @ $695.74 (value: $8,349)


   IWM: $292.17

Submitting virtual BUY order: 10 shares SPY
   Order ID: SHADOW-1784816379394
   Status: filled
   Side: buy
   Quantity: 10

Submitting virtual BUY order: 12 shares QQQ
   Order ID: SHADOW-1784816379396
   Status: filled
   Side: buy
   Quantity: 12

Final Virtual Portfolio:
   Cash: $84,246.78
   SPY: 10 shares @ $740.44 = $7,404.40
   QQQ: 12 shares @ $695.74 = $8,348.82


每次提交后都会打印影子组合，因此从意图到持仓的路径可见，且不承担资金风险。生产缺陷最先暴露在这条路径上，因为策略、券商适配器和风险防护会同时在这条路径上交互。

## 7. 干净关闭

始终要干净地断开连接，以避免下次运行时出现连接问题。规范的关闭流程是实盘可靠性的组成部分，因为陈旧会话和悬挂订阅同样是运维缺陷。

In [19]:
print("\n[OK] Shadow workflow completed and disconnected from IB")


[OK] Shadow workflow completed and disconnected from IB


## 总结

本 notebook 演示了完整的 IB 集成工作流：

1. **连接**：连接到 TWS/Gateway 模拟交易
2. **账户信息**：查询 NLV、现金、持仓
3. **历史数据**：请求 K 线用于策略预热
4. **实时行情流**：订阅 tick 数据并聚合为 K 线
5. **安全交易**：在影子模式下使用 SafeBroker
6. **订单提交**：在 VirtualPortfolio 中跟踪虚拟订单

### 关键要点

- **相同的 Strategy 类** 可在回测和实盘中工作
- **先使用影子模式** - 真实交易前始终先测试
- **SafeBroker** 提供 8 层保护
- **IBBroker** 处理所有 IB 特定细节

### 后续步骤

1. 以影子模式运行 1-2 周
2. 验证信号与回测预期一致
3. 启用模拟交易（`shadow_mode=False`）
4. 在模拟盘监控 2-4 周
5. 逐步以小额持仓过渡到实盘

In [20]:
print("\n" + "=" * 60)
print("DEMO COMPLETE")
print("=" * 60)
print("Paper account readiness: verified")
print(f"Symbols Monitored: {', '.join(SYMBOLS)}")
print("Shadow Mode: ENABLED [OK]")
print("\nThe same MomentumStrategy that runs in backtest")
print("completed the IB shadow workflow without sending an order to the venue.")


DEMO COMPLETE
Paper account readiness: verified
Symbols Monitored: SPY, QQQ, IWM
Shadow Mode: ENABLED [OK]

The same MomentumStrategy that runs in backtest
completed the IB shadow workflow without sending an order to the venue.


## 关键要点

IB 比 Alpaca 承载更多连接性和市场结构复杂性，而上文的执行架构在策略无感知的情况下吸收了所有这些复杂性。一旦连接、预热和影子模式控制成为标准，在券商之间迁移策略就是配置变更，而不是重写。

**下一步**：比较 `04_alpaca_paper_trading_demo` 中更简单的 REST 风格路径，然后转到 `08_pipeline_verification`，检查研究输出和实盘输入是否仍然匹配。